In [1]:
!pip install faker

import numpy as np
import pandas as pd
from faker import Faker
import random
import re

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 8.5 MB/s eta 0:00:00


## Doctor Table

In [2]:
fake = Faker()

In [3]:
# Number of samples
n = 50

# Nominal data: Doctors. Note that these are made less specific for data privacy reasons.
doctor_ids = [f'DR{str(i).zfill(4)}' for i in range(1, n+1)]
np.random.shuffle(doctor_ids)

# Select n unique doctor_ids
doctor_ids_data = doctor_ids[:]
print(len(doctor_ids_data))

# Department names and their respective IDs
department_names = ['Cardiology', 'Neurology', 'Orthopedics', 'Pediatrics', 'Oncology',
                    'Emergency', 'Radiology', 'Dermatology', 'Psychiatry', 'Surgery']

print(len(department_names))

# Create a dictionary that maps department names to department IDs
department_id_mapping = {
    'Cardiology': 'D001',
    'Neurology': 'D002',
    'Orthopedics': 'D003',
    'Pediatrics': 'D004',
    'Oncology': 'D005',
    'Emergency': 'D006',
    'Radiology': 'D007',
    'Dermatology': 'D008',
    'Psychiatry': 'D009',
    'Surgery': 'D010'
}

print(len(department_id_mapping))

# Generate sample doctor names and data
# sample names with given, middle, and last names
sample_names = [f"{fake.first_name()} {fake.first_name()} {fake.last_name()}" for _ in range(n)]
n_points = 5

sample_names = np.array(sample_names)

# Randomly select 5 indices to set to NaN
random_indices = np.random.choice(len(sample_names), n_points, replace=False)

# Set the selected indices to NaN
sample_names[random_indices] = np.nan

print(len(sample_names))


# Generate department data (Nominal Data)
department_data = np.random.choice(department_names, size=n, replace=True)
print(len(department_data))

# Map department names to department IDs using the department_id_mapping dictionary
department_id_data = [department_id_mapping[dept] for dept in department_data]


# Generate joining date (Interval Data)
joining_year = np.random.randint(2000, 2024, n)
joining_month = np.random.randint(1, 13, n)
joining_day = np.random.randint(1, 29, n)
joining_date = [f'{joining_year[i]}-{str(joining_month[i]).zfill(2)}-{str(joining_day[i]).zfill(2)}' for i in range(n)]
print(len(joining_date))
# Generate salary data (Ratio Data)
salary_data = np.random.lognormal(mean=13, sigma=0.5, size=n).astype(int)
print(len(salary_data))
# Generate Emails based on first and last names (first_name.last_name@nhs.com)
def generate_masked_email(name):
    first_name, last_name = name.split()[0], name.split()[-1]

    # Mask part of the first name and last name
    masked_first_name = first_name[0] + '*' * 2
    masked_last_name = last_name[0] + '*' * 2

    # Generate the email
    email = f"{masked_first_name.lower()}.{masked_last_name.lower()}@nhs.com"

    return email


# Function to generate and mask phone numbers, ensuring uniqueness and using only '*' for masking
def generate_unique_masked_phone_numbers(existing_numbers):
    while True:
        # Generate a random phone number (e.g., "+1-123-456-7890")
        phone_number = fake.phone_number()

        # Clean phone number by removing non-numeric characters (remove '-' and '.')
        phone_number_cleaned = re.sub(r'\D', '', phone_number)  # Keep only digits

        # Ensure phone number is 10 digits long after cleaning
        if len(phone_number_cleaned) == 10:
            # Convert to list to easily modify digits
            phone_number_list = list(phone_number_cleaned)

            # Randomly select 3 positions to mask
            mask_positions = random.sample(range(10), 3)

            # Mask those 3 positions with '*'
            for pos in mask_positions:
                phone_number_list[pos] = '*'

            # Join the list back to a string
            phone_number_masked = ''.join(phone_number_list)

            # Format the phone number in the form "XXX-XXX-XXXX" using hyphens
            phone_number_formatted = f"{phone_number_masked[:3]}-{phone_number_masked[3:6]}-{phone_number_masked[6:]}"

            # If this phone number is unique (not already in the list), add it to the existing numbers
            if phone_number_formatted not in existing_numbers:
                existing_numbers.add(phone_number_formatted)
                return phone_number_formatted

# Set to keep track of already generated phone numbers to ensure uniqueness
unique_phone_numbers = set()

# Generate unique masked phone numbers for all doctors
doctor_contact_numbers = [generate_unique_masked_phone_numbers(unique_phone_numbers) for _ in range(n)]



# Generate masked emails based on sample names
Pemails = [generate_masked_email(name) for name in sample_names]
#Pemails = [f"{name.split()[0].lower()}.{name.split()[-1].lower()}@nhs.com" for name in sample_names]
print(len(Pemails))
# Create the DataFrame
df = pd.DataFrame({
    'Doctor_Id': doctor_ids_data,
    'Doctor_Name': sample_names,
    'Joining_Date': joining_date,
    'Salary': salary_data,
    'Department': department_data,
    'Department_Id': department_id_data,  # Department IDs mapped from department names
    'Email': Pemails,
    'Mobile_Number': doctor_contact_numbers
})

# Display the DataFrame (first 5 rows)
print(df.head(10))

# Save to CSV
df.to_csv('Doctor.csv', index=False)


50
10
10
50
50
50
50
50
  Doctor_Id            Doctor_Name Joining_Date   Salary   Department  \
0    DR0034    Anne Patrick Martin   2008-11-23   884850    Emergency   
1    DR0018   Rodney Kelly Bennett   2010-07-27   527346  Dermatology   
2    DR0005  Marco Matthew Hensley   2014-11-20   273615   Psychiatry   
3    DR0038     Sherry Debra Mason   2009-09-12   341910   Pediatrics   
4    DR0039  Shelby Kayla Guerrero   2014-07-15   238159    Neurology   
5    DR0041      Sara Paul Hensley   2010-03-16   299588      Surgery   
6    DR0035      Brenda Janet Hood   2019-08-12  1176309    Neurology   
7    DR0009   Shannon Megan Rivera   2000-06-22   468118  Orthopedics   
8    DR0001                    nan   2009-12-05    86769    Neurology   
9    DR0030   Kimberly Amy Winters   2006-12-03   288401   Psychiatry   

  Department_Id            Email Mobile_Number  
0          D006  a**.m**@nhs.com  *76-*89-*024  
1          D008  r**.b**@nhs.com  480-91*-*59*  
2          D009  m**.h**@

## Department Table

In [9]:
# Function to generate and mask landline phone numbers for departments
def generate_unique_masked_landline_numbers(existing_numbers, country_code="+1", prefix="555"):
        # Generate a landline number starting with country code and fixed prefix
        # The format will be "+1-555-###-####" where "###" and "####" will be random
        phone_number = f"{country_code}-{prefix}-{fake.numerify(text='###-####')}"

        # Ensure the phone number has the expected format: "+1-555-###-####"
        phone_number_cleaned = re.sub(r'\D', '', phone_number)  # Remove non-numeric characters

        # Ensure the number is 10 digits long (after removing the country code)
        if len(phone_number_cleaned) == 11:
            # Convert to list to easily modify digits
            phone_number_list = list(phone_number_cleaned)

            # Randomly select 2 positions to mask
            mask_positions = random.sample(range(6, 10), 2)  # Mask positions from the last part of the number

            # Mask those 2 positions with '*'
            for pos in mask_positions:
                phone_number_list[pos] = '*'

            # Join the list back to a string
            phone_number_masked = ''.join(phone_number_list)

            # Format the phone number back into "+1-555-XXX-XXXX"
            phone_number_formatted = f"{country_code}-{prefix}-{phone_number_masked[3:6]}-{phone_number_masked[6:]}"

            # If this phone number is unique (not already in the list), add it to the existing numbers
            if phone_number_formatted not in existing_numbers:
                existing_numbers.add(phone_number_formatted)
                return phone_number_formatted

# Set to keep track of already generated landline numbers to ensure uniqueness
unique_landline_numbers = set()

# Number of departments (sample size)
n = 10

# Mapping of departments to their respective locations and department IDs (Fixed)
locations = ['Floor 2', 'Wing B, 3rd Floor', 'Outpatient Area', 'Floor 1', 'Wing D, 4th Floor',
             'Ground Floor', 'Floor 3', 'Wing A, 3rd Floor', 'Floor 5', 'Floor 4']

Dept_Id = ['D001','D002','D003','D004','D005','D006','D007','D008','D009','D010']

department_names = ['Cardiology', 'Neurology', 'Orthopedics', 'Pediatrics', 'Oncology',
                    'Emergency', 'Radiology', 'Dermatology', 'Psychiatry', 'Surgery']

# Generate unique landline numbers for each department
landline_numbers = [generate_unique_masked_landline_numbers(unique_landline_numbers) for _ in range(n)]

# Create the DataFrame
df2 = pd.DataFrame({
    'Department': department_names,
    'Department_Id': Dept_Id,
    'Location': locations,
    'Landline_Number': landline_numbers  # Assign generated landline numbers to the column
})

# Print the DataFrame
print(df2)

# Save to CSV
df2.to_csv('Department.csv', index=False)

    Department Department_Id           Location   Landline_Number
0   Cardiology          D001            Floor 2  +1-555-585-57**4
1    Neurology          D002  Wing B, 3rd Floor  +1-555-500-*1*04
2  Orthopedics          D003    Outpatient Area  +1-555-544-**503
3   Pediatrics          D004            Floor 1  +1-555-567-*0*56
4     Oncology          D005  Wing D, 4th Floor  +1-555-502-*66*8
5    Emergency          D006       Ground Floor  +1-555-545-*36*2
6    Radiology          D007            Floor 3  +1-555-591-*7*43
7  Dermatology          D008  Wing A, 3rd Floor  +1-555-504-0**26
8   Psychiatry          D009            Floor 5  +1-555-590-2*9*8
9      Surgery          D010            Floor 4  +1-555-591-83**3


## Patient Table

In [7]:
# Number of samples
n = 700

# Create unique patient IDs
patient_ids = [f'PID{str(i).zfill(4)}' for i in range(1, n + 1)]  # Generate n unique IDs

# Now patient_id_data will have unique patient IDs
patient_id_data = patient_ids

# Generate sample names with given, middle, and last names using Faker
patient_names = [f"{fake.first_name()} {fake.first_name()} {fake.last_name()}" for _ in range(n)]


# To generate 15 duplicate names, we'll select 15 unique names and duplicate them
duplicate_count = 15
duplicate_names = np.random.choice(patient_names, duplicate_count, replace=False)

# Now, we need to duplicate these names a few times.
# For simplicity, let's duplicate each of these 15 names twice (for 30 total duplicates)
duplicated_patient_names = np.tile(duplicate_names, 2)  # Each duplicate name is repeated twice

# Add these duplicates to the original list of unique names
# The total length should now be 700.
final_patient_names = np.concatenate([patient_names, duplicated_patient_names])

# Now, shuffle the final list to distribute duplicates randomly
np.random.shuffle(final_patient_names)


# Ordinal data: Age groups
age_groups2 = ['0-10', '11-18', '19-25', '26-35', '36-45', '46-55', '56-65', '66+']
age_group_data2 = np.random.choice(age_groups2, n, p=[0.15, 0.15, 0.05, 0.1, 0.1, 0.1, 0.2, 0.15])

# Generate Age values based on age group
# We will map each age group to a plausible age range and generate an age accordingly
age_data = []
for group in age_group_data2:
    if group == '0-10':
        age_data.append(np.random.randint(0, 11))  # Age between 0 and 10
    elif group == '11-18':
        age_data.append(np.random.randint(11, 19))  # Age between 11 and 18
    elif group == '19-25':
        age_data.append(np.random.randint(19, 26))  # Age between 19 and 25
    elif group == '26-35':
        age_data.append(np.random.randint(26, 36))  # Age between 26 and 35
    elif group == '36-45':
        age_data.append(np.random.randint(36, 46))  # Age between 36 and 45
    elif group == '46-55':
        age_data.append(np.random.randint(46, 56))  # Age between 46 and 55
    elif group == '56-65':
        age_data.append(np.random.randint(56, 66))  # Age between 56 and 65
    elif group == '66+':
        age_data.append(np.random.randint(66, 102))  # Age between 66 and 101

# Interval data: Admission date
admission_year = np.random.randint(1950, 2025, n)
admission_month = np.random.randint(1, 13, n)
admission_day = np.random.randint(1, 29, n)
admission_date = [f'{admission_year[i]}-{str(admission_month[i]).zfill(2)}-'
                  f'{str(admission_day[i]).zfill(2)}' for i in range(n)]


# Generate Emails based on first and last names (first_name.last_name@nhs.com)
def generate_masked_email(name):
    first_name, last_name = name.split()[0], name.split()[-1]

    # Mask part of the first name and last name
    masked_first_name = first_name[0] + '*' * 2
    masked_last_name = last_name[0] + '*' * 2

    # Generate the email
    emails = f"{masked_first_name.lower()}.{masked_last_name.lower()}@gmail.com"

    return emails

# Generate masked emails based on sample names
emails = [generate_masked_email(name) for name in patient_names]

emails = np.array(emails)

# Function to generate and mask patient mobile numbers, ensuring uniqueness and using only '*' for masking
def generate_unique_masked_patient_mobile_numbers(existing_numbers):
    while True:
        # Generate a random phone number (e.g., "+1-123-456-7890")
        phone_number = fake.phone_number()

        # Clean phone number by removing non-numeric characters (remove '-' and '.')
        phone_number_cleaned = re.sub(r'\D', '', phone_number)  # Keep only digits

        # Ensure phone number is 10 digits long after cleaning
        if len(phone_number_cleaned) == 10:
            # Convert to list to easily modify digits
            phone_number_list = list(phone_number_cleaned)

            # Randomly select 3 positions to mask
            mask_positions = random.sample(range(10), 3)

            # Mask those 3 positions with '*'
            for pos in mask_positions:
                phone_number_list[pos] = '*'

            # Join the list back to a string
            phone_number_masked = ''.join(phone_number_list)

            # Format the phone number in the form "XXX-XXX-XXXX" using hyphens
            phone_number_formatted = f"{phone_number_masked[:3]}-{phone_number_masked[3:6]}-{phone_number_masked[6:]}"

            # If this phone number is unique (not already in the list), add it to the existing numbers
            if phone_number_formatted not in existing_numbers:
                existing_numbers.add(phone_number_formatted)
                return phone_number_formatted

# Set to keep track of already generated patient mobile numbers to ensure uniqueness
unique_patient_mobile_numbers = set()

# Generate unique masked patient mobile numbers for all patients (e.g., 700 patients)
patient_mobile_numbers = [generate_unique_masked_patient_mobile_numbers(unique_patient_mobile_numbers) for _ in range(700)]


n_points = 50

# Randomly select 50 indices to set to NaN
random_indices = np.random.choice(len(emails), n_points, replace=False)

# Set the selected indices to NaN
emails[random_indices] = np.nan

# Create DataFrame with the generated data
df1 = pd.DataFrame({
    'Patient_Id': patient_id_data,
    'Patient_Name': patient_names,
    'Age': age_data,  # Added Age column based on age groups
    'Registration_Date': admission_date,
    'Email': emails,  # Added Email column
    'Mobile_Number': patient_mobile_numbers
})

# Print the first 5 rows of the dataframe
print(df1.head())

# Save to CSV
df1.to_csv('Patient.csv', index=False)

  Patient_Id            Patient_Name  Age Registration_Date  \
0    PID0001  Richard Thomas Bridges   51        1986-08-10   
1    PID0002    Amber Edward Edwards   64        1998-11-11   
2    PID0003     Julia Regina Butler   13        2010-11-13   
3    PID0004    Julia Lauren Kennedy   28        1958-11-19   
4    PID0005     Alexa Kayla Aguilar   12        1994-06-17   

               Email Mobile_Number  
0  r**.b**@gmail.com  830-9**-*996  
1                nan  203-6**-83*5  
2  j**.b**@gmail.com  **6-93*-0204  
3  j**.k**@gmail.com  *60-62*-515*  
4  a**.a**@gmail.com  216-2*2-7**5  


## Appointment Table

In [8]:
# Number of samples
n = 1300

# Generate appointment IDs (Unique for each appointment)
appointment_ids = [f'AID{str(i).zfill(4)}' for i in range(1, n+1)]

# Generate Patient IDs and Details
patient_ids5 = np.random.choice(patient_ids, size=n, replace=True)
#patient_names = np.random.choice(patient_names, size=n, replace=True)

# Generate Emails based on first and last names (first_name.last_name@nhs.com)
def generate_masked_email(name):
    first_name, last_name = name.split()[0], name.split()[-1]

    # Mask part of the first name and last name
    masked_first_name = first_name[0] + '*' * 2
    masked_last_name = last_name[0] + '*' * 2

    # Generate the email
    emails = f"{masked_first_name.lower()}.{masked_last_name.lower()}@gmail.com"

    return emails

# Generate masked emails based on sample names
patient_emails = [generate_masked_email(name) for name in patient_names]

patient_emails = np.array(patient_emails)

n_points = 185

# Randomly select 185 indices to set to NaN
random_indices = np.random.choice(len(patient_emails), n_points, replace=False)

# Set the selected indices to NaN
patient_emails[random_indices] = np.nan

# Generate Doctor IDs and assign them to departments
doctor_ids = np.random.choice(doctor_ids, size=1300, replace=True)
#doctor_names = np.random.choice(sample_names, size=n, replace=True)

# Assign departments to doctors
Dept_Id_list = ['D001','D002','D003','D004','D005','D006','D007','D008','D009','D010']
dept_Id  = np.random.choice(Dept_Id_list, size=n, replace=True)

# Assign emails to patients
#patient_emails = [f"{name.split()[0].lower()}.{name.split()[-1].lower()}@gmail.com" for name in patient_names]# Appointment detail

appointment_year = np.random.randint(2020, 2025, n)
appointment_month = np.random.randint(1, 13, n)
appointment_day = np.random.randint(1, 29, n)
appointment_date = [f'{appointment_year[i]}-{str(appointment_month[i]).zfill(2)}-'
                    f'{str(appointment_day[i]).zfill(2)}' for i in range(n)]

appointment_hour = np.random.randint(9, 18, n)  # Appointments between 9 AM and 6 PM
appointment_minute = np.random.randint(0, 60, n)
appointment_second = np.random.randint(0, 60, n)

appointment_time = [f'{str(appointment_hour[i]).zfill(2)}:{str(appointment_minute[i]).zfill(2)}:{str(appointment_second[i]).zfill(2)}' for i in range(n)]

# Appointment Statuses
appointment_statuses = ['Scheduled', 'Completed', 'Cancelled']
appointment_status = np.random.choice(appointment_statuses, n, p=[0.7, 0.25, 0.05])  # Most appointments are scheduled


# Create the DataFrame
df5 = pd.DataFrame({
    'Appointment_Id': appointment_ids,
    'Patient_Id': patient_ids5,
    'Doctor_Id': doctor_ids,
    'Department_Id': dept_Id,
    'Appointment_Date': appointment_date,
    'Appointment_Time': appointment_time,
    'Appointment_Status': appointment_status
})

# Show the first few rows of the dataframe
print(df5.head())

# Save to CSV
df5.to_csv('Appointments.csv', index=False)


  Appointment_Id Patient_Id Doctor_Id Department_Id Appointment_Date  \
0        AID0001    PID0330    DR0009          D002       2021-02-02   
1        AID0002    PID0138    DR0017          D008       2022-01-17   
2        AID0003    PID0467    DR0031          D009       2020-01-19   
3        AID0004    PID0618    DR0011          D007       2022-08-06   
4        AID0005    PID0394    DR0032          D006       2022-03-18   

  Appointment_Time Appointment_Status  
0         12:45:45          Completed  
1         09:39:29          Scheduled  
2         12:59:02          Scheduled  
3         14:18:41          Scheduled  
4         09:08:55          Scheduled  
